# PRC2 base-editor screen analysis

This notebook reproduces every static figure for the PRC2
base-editor screen manuscript, built on [`be_scan`](https://github.com/liaulab/be-scan).

All screen data is read through `code/data_loading.py` (which reshapes
`PRC2_Screens_Counts_Scores_ZScores.xlsx` and the companion mapping/cluster
workbooks). Shared constants live in `code/config.py`. The PyMOL export and
structural-enrichment analyses live in `code/pymol_export.py` and
`code/structural_enrichment.py`.

Figures are written under `new_outputs/<Category>/`.

## 1. Environment setup

In [ ]:
# @title Install dependencies and detect runtime
# In Colab this installs everything fresh each run; locally it's a no-op if
# the environment is already satisfied. Set RUN_PIP_INSTALL=True to force it.
import os, sys, subprocess

RUN_PIP_INSTALL = False  # set True on a clean Colab runtime

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=False)

if RUN_PIP_INSTALL:
    # Core scientific stack (mirrors environment.yml) + be_scan from GitHub.
    _pip("numpy", "pandas", "scipy", "scikit-learn", "statsmodels",
         "matplotlib", "seaborn", "biopandas", "openpyxl", "nbformat")
    _pip("git+https://github.com/liaulab/be-scan.git")

# Colab detection + Drive mount + chdir into the project folder.
DRIVE_FOLDER = "/content/drive/MyDrive/Liau/PRC2-Data/RevisionAnalysis"  # @param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.chdir(DRIVE_FOLDER)
print("IN_COLAB:", IN_COLAB, "| cwd:", os.getcwd())

# Verify Arial.ttf is present
if not os.path.exists("Arial.ttf"):
    try:
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/liaulab/be-scan/main/be_scan/figure_plot/Arial.ttf",
            "Arial.ttf")
        print("Downloaded Arial.ttf")
    except Exception as e:
        print("Could not fetch Arial.ttf (figures fall back to a system font):", e)
else:
    print("Arial.ttf present")

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.getcwd())
for _m in [m for m in list(sys.modules) if m == "code" or m.startswith("code.")]:
    if getattr(sys.modules[_m], "__file__", "") and "project" not in (sys.modules[_m].__file__ or ""):
        del sys.modules[_m]

from code import config, data_loading, pymol_export, structural_enrichment
from be_scan.figure_plot import *

print("be_scan figure_plot imported")

## 3. Shared configuration

Gene lists, colors, paths and parameters come from `code/config.py`. Figure-specific styling (domain spans, mutation/subunit palettes) that is not in `config.py` is defined here.

In [ ]:
PRC2_GENES = config.PRC2_GENES
genes = ["EZH2", "EED", "SUZ12", "AEBP2", "JARID2"]
xmax_dict = config.PROTEIN_LENGTHS
OUTPUT_DIR = config.OUTPUT_DIR
cm_2_in = 0.393

def outdir(category):
    d = OUTPUT_DIR / category
    d.mkdir(parents=True, exist_ok=True)
    return d

# Mutation-type palette and gene custom_palette for corr scatter
mut_pal = {"Missense": "#FDB462", "No Mutation/Silent": "#80B1D3", "Nonsense": "#C03221"}
custom_palette = {"EZH2": "#B2B0B0", "EED": "#7CCDF4", "SUZ12": "#C19EE0",
                  "Essential": "#FFCC33", "Control": "#A7D5B9", "MTF2": "#90D690"}
color_dict = {"EZH2": "#b2b0b0", "EED": "#7ccdf4", "SUZ12": "#c19ee0",
              "AEBP2": "#1fc1c1", "JARID2": "#ff9999",
              "Essential": "#91caa9", "Control": "#f0c93d"}

subunit_colors = ["#BCE3FA", "#7CCDF4", "#DEC9E9", "#C19EE0", "#D6D6D6", "#B2B0B0",
                  "#90E0EF", "#1FC1C1", "#FFCCCC", "#FF9999", "#B3E8B3", "#90D690"]
subunit_dark_reorder = ["#b2b0b0", "#7ccdf4", "#c19ee0", "#1fc1c1", "#ff9999",
                        "#FFCC33", "#A7D5B9"]
scatter_pal_abe = ["#FDB462", "#3FBF88", "#4A9EF2", "#F2649B", "#B69482", "#A28BE3",
                   "#FFCC33", "#A7D5B9"]
scatter_pal_cbe = ["#FDB462", "#3FBF88", "#4A9EF2", "#E0483C", "#F2649B", "#B69482",
                   "#A28BE3", "#FFCC33", "#A7D5B9"]

domains_all = {
    "AEBP2": [{"description":"ZnF","start":261,"end":285,"color":subunit_colors[6]},
              {"description":"ZnF","start":300,"end":322,"color":subunit_colors[7]},
              {"description":"ZnF","start":329,"end":352,"color":subunit_colors[6]},
              {"description":"KR","start":379,"end":389,"color":subunit_colors[7]},
              {"description":"C2B","start":407,"end":490,"color":subunit_colors[6]},
              {"description":"H3K4D","start":490,"end":504,"color":subunit_colors[7]}],
    "EED": [{"description":"WD40","start":81,"end":125,"color":subunit_colors[0]},
            {"description":"WD40","start":131,"end":176,"color":subunit_colors[1]},
            {"description":"WD40","start":179,"end":219,"color":subunit_colors[0]},
            {"description":"WD40","start":222,"end":264,"color":subunit_colors[1]},
            {"description":"WD40","start":295,"end":332,"color":subunit_colors[0]},
            {"description":"WD40","start":357,"end":397,"color":subunit_colors[1]},
            {"description":"WD40","start":397,"end":438,"color":subunit_colors[0]}],
    "EZH2": [{"description":"SBD","start":14,"end":38,"color":subunit_colors[4]},
             {"description":"EBD","start":38,"end":68,"color":subunit_colors[5]},
             {"description":"SANBAMT","start":68,"end":107,"color":subunit_colors[4]},
             {"description":"SAL","start":107,"end":127,"color":subunit_colors[5]},
             {"description":"SRM","start":127,"end":157,"color":subunit_colors[4]},
             {"description":"SANT1","start":159,"end":250,"color":subunit_colors[5]},
             {"description":"MCSS","start":257,"end":309,"color":subunit_colors[4]},
             {"description":"SANT2","start":428,"end":476,"color":subunit_colors[5]},
             {"description":"CXC","start":503,"end":605,"color":subunit_colors[4]},
             {"description":"SET","start":609,"end":728,"color":subunit_colors[5]}],
    "JARID2": [{"description":"TR","start":140,"end":170,"color":subunit_colors[8]},
               {"description":"EZH1/2 Bind","start":229,"end":349,"color":subunit_colors[9]},
               {"description":"NID","start":349,"end":450,"color":subunit_colors[8]},
               {"description":"JmjN","start":557,"end":598,"color":subunit_colors[9]},
               {"description":"ARID","start":621,"end":713,"color":subunit_colors[8]},
               {"description":"JmjC","start":884,"end":1048,"color":subunit_colors[9]},
               {"description":"ZnF","start":1139,"end":1189,"color":subunit_colors[8]}],
    "SUZ12": [{"description":"ZnB","start":79,"end":106,"color":subunit_colors[2]},
              {"description":"WDB1","start":110,"end":145,"color":subunit_colors[3]},
              {"description":"C2","start":150,"end":365,"color":subunit_colors[2]},
              {"description":"ZnB","start":426,"end":492,"color":subunit_colors[3]},
              {"description":"WDB2","start":514,"end":531,"color":subunit_colors[2]},
              {"description":"VEFS","start":560,"end":682,"color":subunit_colors[3]}],
    "MTF2": [{"description":"Tudor","start":44,"end":101,"color":subunit_colors[10]},
             {"description":"PHD1","start":104,"end":155,"color":subunit_colors[11]},
             {"description":"PHD2","start":203,"end":253,"color":subunit_colors[10]},
             {"description":"EH","start":253,"end":352,"color":subunit_colors[11]},
             {"description":"Chromo-like","start":555,"end":593,"color":subunit_colors[10]}],
}
print("Config loaded. Output root:", OUTPUT_DIR)

## 4. Scatterplots and correlation scatterplots

Per-gene amino-acid-position scatterplots of screen Z-scores then pairwise correlation scatterplots

In [ ]:
# Shared scatter helpers
def scatter_condition(df_ab, ycol, marker_map, ylim, yticks, gene_list, subdir,
                      cell_type, editor_type, tag, figsize=(2.5, 2.8/2.54), suffix=""):
    style = replace(SCATTERSTYLE, hue_col="muttype", color_map=mut_pal,
                    marker_col="editor", marker_map=marker_map)
    axis_base = replace(AXIS, xlabel="Amino Acid Position", ylabel="Base Editing Z-Score",
                        ylim=ylim, yticks=list(yticks))
    for gene in gene_list:
        axis_cfg = replace(axis_base, title=gene, protein_len=xmax_dict[gene])
        out_cfg = replace(OUTPUT, out_type="svg", figsize=figsize, dpi=1200,
                          transparent=True, show=True, save=True,
                          path=str(outdir(subdir) / f"{gene}-{cell_type}-{editor_type}-{ycol}-Scatterplot{suffix}"))
        leg = replace(LEGEND, title="Mutation / Editor", ncol=3, mark_size=2.5, frameon=True,
                      path=str(outdir(subdir) / f"{tag}_Scatter_legend"))
        scatterplot_figure(df_data=df_ab[df_ab["Gene Symbol"] == gene], y_col_list=[ycol],
                           x_col="pos", axis=axis_cfg, style=style,
                           domain=DomainOpts(dom_setting=domains_all[gene]),
                           legend=leg, output=out_cfg)

def build_scatter_df(loader, zcol_a, zcol_c, ycol, ed_a, ed_c):
    base = loader()
    a = base.copy(); a["editor"] = ed_a
    a = a.rename(columns={zcol_a: ycol, "AtoG_pos": "pos", "AtoG_muttype": "muttype"})
    c = base.copy(); c["editor"] = ed_c
    c = c.rename(columns={zcol_c: ycol, "CtoT_pos": "pos", "CtoT_muttype": "muttype"})
    cols = ["Gene Symbol", "pos", ycol, "editor", "muttype"]
    df = pd.concat([a[cols], c[cols]])
    df = df[df["pos"] > 0]
    df = df[df["muttype"].isin(["Missense", "Nonsense", "Silent", "No Mutation"])]
    df.loc[df["muttype"].isin(["Silent", "No Mutation"]), "muttype"] = "No Mutation/Silent"
    return df.reset_index(drop=True)


In [ ]:
df = build_scatter_df(data_loading.load_wt_screen,
                      "WT_LOF_dropout_ABE_zscore", "WT_LOF_dropout_CBE6b_zscore",
                      "DMSO-D0-Z", "ABE", "CBE6b")
scatter_condition(df, "DMSO-D0-Z", {"ABE": "o", "CBE6b": "D"}, (-10, 7.5),
                  np.linspace(-10, 7.5, 8), genes, "Scatterplots",
                  "Karpas422_WT", "ABE-CBE6b", "WT", figsize=(4.3/2.54, 2.8/2.54))

skip_combos = {("PRC2i-D0-Z", ""), ("DMSO-D0-Z", ""), ("DMSO-D0-Z", "5"), ("DMSO-PRC2i-Z", "5")}
for zbase, ycol, ylim, yt, tag in [
        ("Q575R_LOF_dropout", "PRC2i-D0-Z", (-9, 9), np.linspace(-9, 9, 7), "Q575R_Dropout"),
        ("Q575R_LOF_dmso_d0", "DMSO-D0-Z", (-6, 9), np.linspace(-6, 9, 6), "Q575R_Dropout"),
        ("Q575R_LOF_positiveselection", "DMSO-PRC2i-Z", (-6, 9), np.linspace(-6, 9, 6), "Q575R")]:
    df = build_scatter_df(data_loading.load_q575r_screen,
                          f"{zbase}_ABE_zscore", f"{zbase}_CBE39_zscore", ycol, "ABE", "CBE3")
    for suffix, fs in [("", (2.5, 2.8/2.54)), ("5", (4.3/2.54, 2.8/2.54))]:
        if (ycol, suffix) in skip_combos:
            continue
        scatter_condition(df, ycol, {"ABE": "o", "CBE3": "D"}, ylim, yt, genes, "Scatterplots",
                          "Karpas422_Q575R", "ABE-CBE3", tag, figsize=fs, suffix=suffix)
print("WT / Q575R scatterplots done")


In [ ]:
# Stability scatter
def build_stability_scatter(zcol_ezh2, zcol_eed, pos_col, mut_col, ycol):
    base = data_loading.load_stability_screen()
    e = base.copy(); e["celltype"] = "GFP_EZH2_K562"
    e = e.rename(columns={zcol_ezh2: ycol, pos_col: "pos", mut_col: "muttype"})
    d = base.copy(); d["celltype"] = "GFP_EED_K562"
    d = d.rename(columns={zcol_eed: ycol, pos_col: "pos", mut_col: "muttype"})
    cols = ["Gene Symbol", "pos", ycol, "celltype", "muttype"]
    df = pd.concat([e[cols], d[cols]])
    df = df[df["pos"] > 0]
    df = df[df["muttype"].isin(["Missense", "Nonsense", "Silent", "No Mutation"])]
    df.loc[df["muttype"].isin(["Silent", "No Mutation"]), "muttype"] = "No Mutation/Silent"
    return df.reset_index(drop=True)

for tag, ez, ee, pcol, mcol, ylim, yt in [
    ("ABE", "Abundance_GFP-EZH2_ABE_zscore", "Abundance_GFP-EED_ABE_zscore",
     "AtoG_pos", "AtoG_muttype", (-10, 10), np.linspace(-10, 10, 5)),
    ("CBE", "Abundance_GFP-EZH2_CBE6b_zscore", "Abundance_GFP-EED_CBE6b_zscore",
     "CtoT_pos", "CtoT_muttype", (-8, 16), np.linspace(-8, 16, 4))]:
    df = build_stability_scatter(ez, ee, pcol, mcol, "lowGFP-unsorted-Z")
    style = replace(SCATTERSTYLE, hue_col="muttype", color_map=mut_pal,
                    marker_col="celltype",
                    marker_map={"GFP_EZH2_K562": "o", "GFP_EED_K562": "D"})
    axis_base = replace(AXIS, xlabel="Amino Acid Position", ylabel="Base Editing Z-Score",
                        ylim=ylim, yticks=list(yt))
    for gene in genes:
        out_cfg = replace(OUTPUT, out_type="svg", figsize=(2.5, 2.8/2.54), save=True, show=True,
                          path=str(outdir("Scatterplots") /
                                   f"{gene}-GFP_EZH2_EED_K562-{tag}-lowGFP-unsorted-Z-Scatterplot"))
        scatterplot_figure(df_data=df[df["Gene Symbol"] == gene], y_col_list=["lowGFP-unsorted-Z"],
                           x_col="pos", axis=replace(axis_base, title=gene, protein_len=xmax_dict[gene]),
                           style=style, domain=DomainOpts(dom_setting=domains_all[gene]),
                           legend=replace(LEGEND, title="Mutation / Cell Type", ncol=3,
                                          mark_size=2.5, frameon=True,
                                          path=str(outdir("Scatterplots") / f"Stability_{tag}_legend")),
                           output=out_cfg)
print("Stability scatterplots done")

In [ ]:
# MTF2 scatter
mtf2 = data_loading.load_mtf2_screen()
for ycol, zcol, cell_type, ylim, yt in [
    ("PRC2i-D0-Z", "Q575R_PRC2i_D0_Wk3_zscore", "Karpas422_Q575R", (-6, 6), np.linspace(-6, 6, 5)),
    ("DMSO-D0-Z", "Q575R_DMSO_D0_Wk3_zscore", "Karpas422_Q575R", (-6, 6), np.linspace(-6, 6, 5)),
    ("DMSO-PRC2i-Z", "Q575R_DMSO_PRC2i_Wk3_zscore", "Karpas422_Q575R", (-8, 8), np.linspace(-8, 8, 5)),
    ("DMSO-D0-Z", "WT_DMSO_D0_Wk3_zscore", "Karpas422_WT", (-6, 6), np.linspace(-6, 6, 5))]:
    d = mtf2.copy(); d["editor"] = "ABE"
    d = d.rename(columns={zcol: ycol, "aa_pos": "pos", "AtoG_muttype": "muttype"})
    d = d[d["pos"] > 0]
    d = d[d["muttype"].isin(["Missense", "Nonsense", "Silent", "No Mutation"])]
    d.loc[d["muttype"].isin(["Silent", "No Mutation"]), "muttype"] = "No Mutation/Silent"
    leg_name = "Q575R_MTF2_Scatter_legend" if "Q575R" in cell_type else "WT_MTF2_Scatter_legend"
    scatterplot_figure(
        df_data=d[d["Gene Symbol"] == "MTF2"], y_col_list=[ycol], x_col="pos",
        axis=replace(AXIS, title="MTF2", xlabel="Amino Acid Position",
                     ylabel="Base Editing Z-Score", ylim=ylim, yticks=list(yt),
                     protein_len=xmax_dict["MTF2"]),
        style=replace(SCATTERSTYLE, hue_col="muttype", color_map=mut_pal,
                      marker_col="editor", marker_map={"ABE": "o"}),
        domain=DomainOpts(dom_setting=domains_all["MTF2"]),
        legend=replace(LEGEND, title="Mutation / Editor", ncol=3, mark_size=2.5, frameon=True,
                       path=str(outdir("Scatterplots") / leg_name)),
        output=replace(OUTPUT, out_type="svg", figsize=(2.5, 2.8/2.54), save=True, show=True,
                       path=str(outdir("Scatterplots") /
                                f"MTF2-{cell_type}-ABE-{ycol}-Scatterplot")))
print("MTF2 scatterplots done")


In [ ]:
# Correlation scatterplots
corr_style = replace(SCATTERSTYLE,
                     transparent_kws={"edgecolor": "black", "alpha": 0.25, "s": 4, "linewidth": 0.1},
                     opaque_kws={"edgecolor": "black", "alpha": 1.0, "s": 4, "linewidth": 0.1},
                     line_kws={"color": "k", "ls": "--", "lw": 0.5})
leg_order = ["EZH2", "EED", "SUZ12", "Essential", "Control"]

# Q575R DMSO-PRC2i vs PRC2i-D0
q = data_loading.load_q575r_screen()
for ed, en, cy in [("ABE", "ABE", (-10, 10)), ("CBE39", "CBE", (-6, 6))]:
    d = q.copy()
    d = d.rename(columns={f"Q575R_LOF_positiveselection_{ed}_zscore": "DMSO-PRC2i-Z",
                          f"Q575R_LOF_dropout_{ed}_zscore": "PRC2i-D0-Z"})
    d["mut_type"] = d["AtoG_muttype"] if ed == "ABE" else d["CtoT_muttype"]
    d = d[~d["Gene Symbol"].isin(["AEBP2", "JARID2"])]
    axis_style = replace(AXIS, xlim=(-6, 9), ylim=cy, xticks=[-6, -3, 0, 3, 6, 9],
                         yticks=list(np.linspace(cy[0], cy[1], 5)),
                         title="DMSO-PRC2i vs PRC2i-D0",
                         xlabel="DMSO-PRC2i Score", ylabel="PRC2i-D0 Score")
    out_style = replace(OUTPUT, figsize=(2.8/2.54, 2.8/2.54), save=True, rasterize=True,
                        path=str(outdir("Scatterplots") /
                                 f"DMSO-PRC2i-PRC2i-D0-{en}-correlation_scatterplot"))
    correlation_scatterplot_figure(d, "DMSO-PRC2i-Z", "PRC2i-D0-Z", palette=custom_palette,
                                   legend_order=leg_order, hue_col="Gene Symbol",
                                   axis=axis_style, style=corr_style, output=out_style)

# GFP-EZH2 vs GFP-EED stability
s = data_loading.load_stability_screen()
for ed, en in [("ABE", "ABE"), ("CBE6b", "CBE")]:
    d = s.copy()
    d = d.rename(columns={f"Abundance_GFP-EZH2_{ed}_zscore": "GFP EZH2 lowGFP-unsort",
                          f"Abundance_GFP-EED_{ed}_zscore": "GFP EED lowGFP-unsort"})
    d["mut_type"] = d["AtoG_muttype"] if ed == "ABE" else d["CtoT_muttype"]
    d = d[~d["Gene Symbol"].isin(["AEBP2", "JARID2"])]
    axis_style = replace(AXIS, xlim=(-8, 12), ylim=(-10, 15), xticks=[-8, -4, 0, 4, 8, 12],
                         yticks=[-10, -5, 0, 5, 10, 15], title="GFP-EZH2 vs GFP-EED",
                         xlabel="GFP-EZH2 Score", ylabel="GFP-EED Score")
    out_style = replace(OUTPUT, figsize=(2.8/2.54, 2.8/2.54), save=True, rasterize=True,
                        path=str(outdir("Scatterplots") /
                                 f"GFP-EZH2-GFP-EED-{en}-correlation_scatterplot"))
    correlation_scatterplot_figure(d, "GFP EZH2 lowGFP-unsort", "GFP EED lowGFP-unsort",
                                   palette=custom_palette, legend_order=["EZH2", "EED", "SUZ12"],
                                   hue_col="Gene Symbol", axis=axis_style, style=corr_style,
                                   output=out_style)

print("Correlation scatterplots done")

In [ ]:
# MTF2-screen correlation scatterplots
mtf2 = data_loading.load_mtf2_screen().copy()
mtf2["mut_type"] = mtf2["AtoG_muttype"].replace({"Silent": "No Mutation/Silent",
                                                "No Mutation": "No Mutation/Silent"})
mut_leg = ["Missense", "No Mutation/Silent"]
m = mtf2[mtf2["mut_type"].isin(mut_leg)].rename(columns={
    "WT_DMSO_D0_Wk3_zscore": "Parental DMSO-D0-Z",
    "Q575R_DMSO_D0_Wk3_zscore": "Q575R DMSO-D0-Z",
    "Q575R_DMSO_PRC2i_Wk3_zscore": "Q575R DMSO-PRC2i-Z"}).reset_index(drop=True)
for xcol, ycol, lim, ticks, title, xl, yl, fname in [
        ("Parental DMSO-D0-Z", "Q575R DMSO-D0-Z", (-5, 5), [-5, 0, 5], "Parental vs Q575R",
         "Parental sgRNA Score", "Q575R sgRNA Score", "Parental-Q575R-ABE")]:
    correlation_scatterplot_figure(
        m, xcol, ycol, palette=mut_pal, legend_order=mut_leg, hue_col="mut_type",
        axis=replace(AXIS, xlim=lim, ylim=lim, xticks=ticks, yticks=ticks, title=title,
                     xlabel=xl, ylabel=yl),
        style=corr_style,
        output=replace(OUTPUT, figsize=(2.8/2.54, 2.8/2.54), save=True, rasterize=True,
                       path=str(outdir("Scatterplots") / f"{fname}-correlation_scatterplot")))
print("MTF2 correlation scatterplots done")


## 5. Boxplots and strip plots

Editor×gene and editor×mutation-type distributions

In [ ]:
# Boxplot data prep helper
box_cols = ["mut_type", "Gene Symbol", "Plot Col", "Editor", "aa_pos"]

def prep_box(base, editor, plot_col, pos_col, mut_col):
    df = base.copy()
    df["Editor"] = editor
    df["Plot Col"] = df[plot_col]
    df["mut_type"] = df[mut_col].astype(str)
    df["aa_pos"] = df[pos_col]
    return df[box_cols]

gene_order_q = ["EZH2", "EED", "SUZ12", "AEBP2", "JARID2", "Essential Control", "NT Control"]
mut_order_q = ["Missense", "Silent", "No Mutation", "Nonsense", "Splice", "UTR", "Intron",
               "Essential Control", "NT Control"]

def combine_box(frames):
    df = pd.concat(frames, ignore_index=True)
    df["mut_type"] = df["mut_type"].replace({"Splice-donor": "Splice", "Splice-acceptor": "Splice"})
    df.loc[df["Gene Symbol"] == "Control", ["mut_type", "Gene Symbol"]] = "NT Control"
    df.loc[df["Gene Symbol"] == "Essential", ["mut_type", "Gene Symbol"]] = "Essential Control"
    df = df[df["mut_type"].str.lower() != "nan"]
    df["Editor-Gene"] = df["Editor"] + "-" + df["Gene Symbol"]
    df["Editor-Mutation"] = df["Editor"] + "-" + df["mut_type"]
    return df

def run_boxplots(df, editors, gene_pal, mut_pal_list, title, fname_gene, fname_mut,
                 ylim, yticks, figsize):
    lab_gene = [f"{e}-{g}" for e in editors for g in gene_order_q]
    lab_mut = [f"{e}-{m}" for e in editors for m in mut_order_q]
    for e in editors:
        if f"{e}-Nonsense" in lab_mut and e in ("ABE", "EZH2 ABE", "EED ABE"):
            lab_mut.remove(f"{e}-Nonsense")
    boxplot_figure(df=df, x_col="Editor-Gene", y_col="Plot Col", lab_order=lab_gene,
                   palette=gene_pal, axis=replace(AXIS, xlabel="Editor-Gene",
                   ylabel="Base Editing Z-Score", title=f"{title}, By Gene",
                   ylim=ylim, yticks=yticks),
                   output=replace(OUTPUT, figsize=figsize, save=True, show=True,
                   path=str(outdir("Boxplots") / fname_gene)))
    boxplot_figure(df=df, x_col="Editor-Mutation", y_col="Plot Col", lab_order=lab_mut,
                   palette=mut_pal_list, axis=replace(AXIS, xlabel="Editor-Mutation",
                   ylabel="Base Editing Z-Score", title=f"{title}, By Mutation",
                   ylim=ylim, yticks=yticks),
                   output=replace(OUTPUT, figsize=figsize, save=True, show=True,
                   path=str(outdir("Boxplots") / fname_mut)))

In [ ]:
# WT boxplots
w = data_loading.load_wt_screen()
frames = [prep_box(w, "ABE", "WT_LOF_dropout_ABE_zscore", "AtoG_pos", "AtoG_muttype"),
          prep_box(w, "CBE39", "WT_LOF_dropout_CBE39_zscore", "CtoT_pos", "CtoT_muttype"),
          prep_box(w, "CBE6", "WT_LOF_dropout_CBE6b_zscore", "CtoT_pos", "CtoT_muttype")]
run_boxplots(combine_box(frames), ["ABE", "CBE39", "CBE6"], subunit_dark_reorder,
             scatter_pal_abe + scatter_pal_cbe + scatter_pal_cbe, "WT",
             "WT_Boxplot_ByGene", "WT_Boxplot_ByMutation", (-6, 4), [-6, -4, -2, 0, 2, 4],
             (17/2*cm_2_in, 1.3))

# Q575R boxplots
q = data_loading.load_q575r_screen()
frames = [prep_box(q, "ABE", "Q575R_LOF_positiveselection_ABE_zscore", "AtoG_pos", "AtoG_muttype"),
          prep_box(q, "CBE39", "Q575R_LOF_positiveselection_CBE39_zscore", "CtoT_pos", "CtoT_muttype")]
run_boxplots(combine_box(frames), ["ABE", "CBE39"], subunit_dark_reorder,
             scatter_pal_abe + scatter_pal_cbe, "Q575R",
             "Q575R_Boxplot_ByGene", "Q575R_Boxplot_ByMutation", (-4, 5),
             list(np.linspace(-4, 5, 10)), (17/2*cm_2_in, 1.3))

# Stability boxplots
s = data_loading.load_stability_screen()
frames = [
    prep_box(s, "EZH2 ABE", "Abundance_GFP-EZH2_ABE_zscore", "AtoG_pos", "AtoG_muttype"),
    prep_box(s, "EZH2 CBE6", "Abundance_GFP-EZH2_CBE6b_zscore", "CtoT_pos", "CtoT_muttype"),
    prep_box(s, "EED ABE", "Abundance_GFP-EED_ABE_zscore", "AtoG_pos", "AtoG_muttype"),
    prep_box(s, "EED CBE6", "Abundance_GFP-EED_CBE6b_zscore", "CtoT_pos", "CtoT_muttype")]
run_boxplots(combine_box(frames), ["EZH2 ABE", "EZH2 CBE6", "EED ABE", "EED CBE6"],
             subunit_dark_reorder, scatter_pal_abe + scatter_pal_cbe + scatter_pal_abe + scatter_pal_cbe,
             "Stability", "Stability_Boxplot_ByGene", "Stability_Boxplot_ByMutation",
             (-4, 6), [-4, -2, 0, 2, 4, 6], (19/2*cm_2_in, 1.1))
print("Boxplots done")

In [ ]:
# Jitter / strip plots
q = data_loading.load_q575r_screen()[["Gene Symbol", "AtoG_pos", "sgRNA sequence",
                                       "Q575R_LOF_positiveselection_ABE_zscore"]]
q = q.rename(columns={"Q575R_LOF_positiveselection_ABE_zscore": "DMSO-PRC2i-Z"})
w = data_loading.load_wt_screen()[["Gene Symbol", "sgRNA sequence", "WT_LOF_dropout_ABE_zscore"]]
w = w.rename(columns={"WT_LOF_dropout_ABE_zscore": "DMSO-D0-Z"})
lof = q.merge(w, on=["Gene Symbol", "sgRNA sequence"]).dropna(subset=["DMSO-PRC2i-Z", "DMSO-D0-Z"])
lof["quartile"] = pd.qcut(lof["DMSO-D0-Z"], q=4, labels=False) + 1
jitterbox_kdeplot_figure(lof, xcol="quartile", ycol="DMSO-PRC2i-Z", huecol="Gene Symbol",
                         color_dict=color_dict,
                         axis=replace(AXIS, title="PRC2", ylabel="Q575R Score",
                                      xlabel="Quartile of WT Dropout", ylim=(-6, 10),
                                      yticks=[-6, -4, -2, 0, 2, 4, 6, 8, 10]),
                         output=replace(OUTPUT, save=True, show=True, figsize=(3/2.54, 3/2.54),
                                        path=str(outdir("Boxplots") / "PRC2_Quartile_WT_Q575R")))
print("Jitter/strip plots done")

In [ ]:
# WT PRC2i-resistance jitter boxplot
w = data_loading.load_wt_screen()
res = pd.concat([
    w[["Gene Symbol", "sgRNA sequence", "WT_LOF_positiveselection_ABE_zscore", "AtoG_mutations"]]
     .rename(columns={"WT_LOF_positiveselection_ABE_zscore": "PRC2i-D0-Z",
                      "AtoG_mutations": "mutations"}).assign(editor="ABE"),
    w[["Gene Symbol", "sgRNA sequence", "WT_LOF_positiveselection_CBE6b_zscore", "CtoT_mutations"]]
     .rename(columns={"WT_LOF_positiveselection_CBE6b_zscore": "PRC2i-D0-Z",
                      "CtoT_mutations": "mutations"}).assign(editor="CBE6b"),
]).dropna(subset=["PRC2i-D0-Z"]).reset_index(drop=True)
jitterbox_kdeplot_figure(
    res, xcol="editor", ycol="PRC2i-D0-Z", huecol="Gene Symbol", color_dict=color_dict,
    kde=False, threshold=6.2,
    style=replace(JITTERPLOTSTYLE, stripplot_kws={"alpha": 0.75, "size": 2.5, "linewidth": 0,
                  "zorder": 0, "jitter": True, "dodge": False, "rasterized": True}),
    axis=replace(AXIS, title="WT Dropout", ylabel="sgRNA Score", xlabel="Base Editor",
                 ylim=(-8, 12), yticks=[-8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12]),
    output=replace(OUTPUT, save=True, show=True, figsize=(3/2.54, 4/2.54),
                   path=str(outdir("Boxplots") / "WT_Resistance")))

# MTF2-screen boxplots by mutation type — NB1 cells 21/22 (old Figure3_Boxplot).
mtf2 = data_loading.load_mtf2_screen().copy()
lab_mut1 = [f"ABE-{m}" for m in mut_order_q if m != "Nonsense"]
for zcol, title, fname in [
        # PRC2i-D0 dropped
        ("Q575R_DMSO_D0_Wk3_zscore", "Q575R DMSO-D0", "MTF2_Q575R_DMSOD0_Boxplot_ByMutation"),
        ("Q575R_DMSO_PRC2i_Wk3_zscore", "Q575R DMSO-PRC2i", "MTF2_Q575R_DMSOPRC2i_Boxplot_ByMutation"),
        ("WT_DMSO_D0_Wk3_zscore", "WT DMSO-D0", "MTF2_WT_DMSOD0_Boxplot_ByMutation")]:
    dfm = combine_box([prep_box(mtf2, "ABE", zcol, "aa_pos", "AtoG_muttype")])
    boxplot_figure(df=dfm, x_col="Editor-Mutation", y_col="Plot Col", lab_order=lab_mut1,
                   palette=scatter_pal_abe,
                   axis=replace(AXIS, xlabel="Editor-Mutation", ylabel="Base Editing Z-Score",
                                title=f"{title}, By Mutation", ylim=(-4, 5),
                                yticks=list(np.linspace(-4, 5, 10))),
                   output=replace(OUTPUT, figsize=(4.5*cm_2_in, 1.3), save=True, show=True,
                                  path=str(outdir("Boxplots") / fname)))
print("WT resistance + MTF2 boxplots done")


In [ ]:
# Quantile-stratified boxplot/stripplot
abe = data_loading.load_sgrna_level("ABE"); abe["editor"] = "ABE"
cbe = data_loading.load_sgrna_level("CBE"); cbe["editor"] = "CBE"
both = pd.concat([abe, cbe])
qaxis = replace(AXIS, ylim=(-3.0, 12), yticks=[-3, 0, 3, 6, 9, 12])
for name, dd in [("merged", both)]:
    cmp = structural_enrichment.plot_model_quantile_boxplot_v2(
        dd, "GFP-MAX", str(outdir("Boxplots") / f"Quantile_Box_plot_{name}"),
        axis=qaxis, width_cm=7.3)
print("Quantile boxplots done")

## 6. Lollipop plots

Per-residue hit lollipops

In [ ]:
# Lollipop helper
LOL_FIGSIZE = (2.5, 0.75)
LOL_SPLIT_FIGSIZE = (2.5*1.063, 1)

def lollipop(gene, df_pos, pos_col, pos_thr, pos_cat, pos_colors,
             df_neg, neg_col, neg_thr, neg_cat, neg_colors, title, path,
             ylim=None, yticks=None, split=None, xcol="pos",
             figsize=None, height_ratios=(0.5, 0.001, 1)):
    """split=None -> lollipop_figure(ylim, yticks);
    split=((ylim_top, yticks_top), (ylim_bot, yticks_bot)) -> lollipop_split_figure."""
    gl = xmax_dict[gene]
    axis = replace(AXIS, title=title, xlim=(1-10, gl+10), xticks=[1, gl], ylabel="Z-Score",
                   xlabel="Amino Acid Position", linewidth=0.5)
    common = dict(domain=replace(DOMAIN, dom_setting=domains_all[gene], default_alpha=0.25),
                  style=replace(LOLLIPOPSTYLE, marker_rasterized=True))
    if split is None:
        lollipop_figure(df_pos, xcol, pos_col, pos_thr, pos_cat, pos_colors,
                        df_neg, xcol, neg_col, neg_thr, neg_cat, neg_colors,
                        axis=replace(axis, ylim=ylim, yticks=list(yticks)),
                        output=replace(OUTPUT, figsize=figsize or LOL_FIGSIZE, save=True,
                                       path=str(path)), **common)
    else:
        (yl_t, yt_t), (yl_b, yt_b) = split
        lollipop_split_figure(df_pos, xcol, pos_col, pos_thr, pos_cat, pos_colors, yl_t, yt_t,
                              df_neg, xcol, neg_col, neg_thr, neg_cat, neg_colors, yl_b, yt_b,
                              height_ratios=list(height_ratios), axis=axis,
                              output=replace(OUTPUT, figsize=figsize or LOL_SPLIT_FIGSIZE,
                                             save=True, path=str(path)), **common)

# Q575R LOF lollipops (Q575R DMSO-PRC2i enrich on top, WT DMSO-D0 dropout below)
q = data_loading.load_q575r_screen(); w = data_loading.load_wt_screen()
def _lol_prep(df, zcol, cat, pos="AtoG_pos", neg=False, drop_nonsense=False,
              mut="AtoG_muttype"):
    d = df.copy()
    if drop_nonsense:
        d = d[d[mut] != "Nonsense"]
    d["pos"] = d[pos]; d["editor"] = cat; d = d[d["pos"] > 0]
    d["Plot"] = (-1 if neg else 1) * d[zcol]
    return d[["sgRNA ID", "pos", "Plot", "editor", "Gene Symbol"]]

lof_sets = {
    "ABE": (_lol_prep(q, "Q575R_LOF_positiveselection_ABE_zscore", "ABE"),
            _lol_prep(w, "WT_LOF_dropout_ABE_zscore", "ABE"),
            ["#feb1c6"], ["#5aa9e6"], (-10, 10), np.linspace(-10, 10, 5),
            (((0, 10), [0, 5, 10]), ((-10, 0), [-10, -5, 0]))),
    "CBE": (_lol_prep(q, "Q575R_LOF_positiveselection_CBE39_zscore", "CBE39", pos="CtoT_pos",
                      drop_nonsense=True, mut="CtoT_muttype"),
            pd.concat([_lol_prep(w, "WT_LOF_dropout_CBE39_zscore", "CBE39", pos="CtoT_pos",
                                 drop_nonsense=True, mut="CtoT_muttype"),
                       _lol_prep(w, "WT_LOF_dropout_CBE6b_zscore", "CBE6", pos="CtoT_pos",
                                 drop_nonsense=True, mut="CtoT_muttype")]),
            ["#feb1c6"], ["#afffb9", "#5aa9e6"], (-10, 10), np.linspace(-10, 10, 5),
            (((0, 8), [0, 4, 8]), ((-8, 0), [-8, -4, 0]))),
}

for ed, (dpos, dneg, pc, nc, yl, yt, split) in lof_sets.items():
    for gene in genes:
        args = (gene, dpos[dpos["Gene Symbol"] == gene], "Plot", 2, "editor", pc,
                dneg[dneg["Gene Symbol"] == gene], "Plot", -2, "editor", nc, f"{gene} - {ed}")
        base = outdir("Lollipops") / f"Q575R_{ed}_{gene}_Lollipop"
        lollipop(*args, path=f"{base}_Split", split=split)
print("Q575R LOF lollipops done")


In [ ]:
# LOF-vs-Stability lollipops
q = data_loading.load_q575r_screen(); w = data_loading.load_wt_screen()
s = data_loading.load_stability_screen()
cols = ["sgRNA ID", "pos", "screen", "Plot", "Gene Symbol"]
def _p(df, zcol, screen, neg=False, pos="AtoG_pos"):
    d = df.copy(); d["pos"] = d[pos]; d = d[d["pos"] > 0]
    d["screen"] = screen; d["Plot"] = (-1 if neg else 1) * d[zcol]
    return d[cols]

stab_sets = {
    "ABE": (pd.concat([_p(q, "Q575R_LOF_positiveselection_ABE_zscore", "Q575R"),
                       _p(w, "WT_LOF_dropout_ABE_zscore", "WT", neg=True)]),
            ["#ff5a5a", "#feb1c6"],
            pd.concat([_p(s, "Abundance_GFP-EZH2_ABE_zscore", "GFP-EZH2", neg=True),
                       _p(s, "Abundance_GFP-EED_ABE_zscore", "GFP-EED", neg=True)]),  # E-09
            ["#afffb9", "#5aa9e6"], (-12, 12), np.linspace(-12, 12, 5),
            (((0, 12), [0, 6, 12]), ((-12, 0), [-12, -6, 0]))),
    "CBE": (pd.concat([_p(q, "Q575R_LOF_positiveselection_CBE39_zscore", "Q575R CBE39", pos="CtoT_pos"),
                       _p(w, "WT_LOF_dropout_CBE39_zscore", "WT CBE39", neg=True, pos="CtoT_pos"),
                       _p(w, "WT_LOF_dropout_CBE6b_zscore", "WT CBE6", neg=True, pos="CtoT_pos")]),
            ["#ff5a5a", "#ffd55a", "#feb1c6"],
            pd.concat([_p(s, "Abundance_GFP-EZH2_CBE6b_zscore", "GFP-EZH2", neg=True, pos="CtoT_pos"),
                       _p(s, "Abundance_GFP-EED_CBE6b_zscore", "GFP-EED", neg=True, pos="CtoT_pos")]),
            ["#afffb9", "#5aa9e6"], (-16, 8), [-16, -8, 0, 8],
            (((0, 8), [0, 8]), ((-16, 0), [-16, -8, 0]))),
}

for ed, (dt, tc, db, bc, yl, yt, split) in stab_sets.items():
    for gene in genes:
        args = (gene, dt[dt["Gene Symbol"] == gene], "Plot", 2, "screen", tc,
                db[db["Gene Symbol"] == gene], "Plot", -2, "screen", bc,
                f"{gene} - LOF vs Stability - {ed}")
        base = outdir("Lollipops") / f"LOF_Stability_{ed}_{gene}_Lollipop"
        lollipop(*args, path=f"{base}_Split", split=split)
print("LOF-vs-Stability lollipops done")


## 7. Clustered heatmaps

k-means clustering (k=6) of multi-screen hit patterns, ABE and CBE (NB1 cells
50-59): the hit-guide heatmaps at a Z-score cutoff of 3 (`*_K6_Raw_Cutoff3`), the
wider cutoff-2 heatmaps (`*_K6_Raw`, `*_K6`; the old files carry no cutoff in
their name but contain exactly the |Z| > 2 guides), and the ABE per-cluster
subset heatmaps. Rows with missing screen values are dropped before clustering
(errors.md E-19).

k-means uses `config.KMEANS_RANDOM_STATE` (seed chosen by grid search against
`PRC2_Screens_KMeans_Clusters.xlsx`; errors.md "K-means seed lock-in"). Because
k-means label integers are arbitrary, the cutoff-3 labels are then renamed to
the reference's integer labels by maximum-overlap matching (the partition is
not changed), so NB1's `cluster_order` / colour palette keep meaning the same
clusters.


In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score

screen_order = ["Q575R-LOF-Enrich", "WT-LOF-Deplete", "GFP-EZH2-Enrich", "GFP-EED-Enrich"]
gene_palette = {g: hex_to_rgb(c) for g, c in [
    ("EZH2", "#B2B0B0"), ("EED", "#7CCDF4"), ("SUZ12", "#C19EE0"),
    ("AEBP2", "#1FC1C1"), ("JARID2", "#FF9999")]}
gene_list = ["EZH2", "EED", "SUZ12", "AEBP2", "JARID2"]
tab20 = ["#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78", "#2ca02c", "#98df8a"]
ROMAN = ["I", "II", "III", "IV", "V", "VI"]
K = config.KMEANS_K

def heatmap_input(cols, hit_thr=None):
    """Merge the four screens (NB1 cells 53/57); optional +/-hit_thr hit filter."""
    q = data_loading.load_q575r_screen().rename(columns={cols[0]: "Q575R-LOF-Enrich"})
    w = data_loading.load_wt_screen().rename(columns={cols[1]: "WT-LOF-Deplete"})
    s = data_loading.load_stability_screen().rename(
        columns={cols[2]: "GFP-EZH2-Enrich", cols[3]: "GFP-EED-Enrich"})
    keys = ["sgRNA ID", "sgRNA sequence"]
    m = pd.merge(q[keys + ["Gene Symbol", "Q575R-LOF-Enrich"]], w[keys + ["WT-LOF-Deplete"]],
                 on=keys, how="inner")
    m = pd.merge(m, s[keys + ["GFP-EZH2-Enrich", "GFP-EED-Enrich"]], on=keys, how="inner")
    if hit_thr is not None:
        m = m[((m[screen_order] > hit_thr) | (m[screen_order] < -hit_thr)).any(axis=1)]
    m = m[m["Gene Symbol"].isin(gene_list)]
    return m.set_index("sgRNA ID")[screen_order].dropna().sort_index()

def align_to_reference(idf, clusters, editor, cluster_order):
    """Rename k-means label integers to the reference workbook's integers.
    Reference roman numeral r_i is NB1's cluster_order[i] (verified against the
    old Drive CSVs). Returns (renamed labels, ARI vs reference)."""
    ref = data_loading.load_kmeans_clusters(editor).set_index("sgRNA ID")["clusters"]
    ref = ref.map(dict(zip(ROMAN, cluster_order)))
    lab = pd.Series(clusters, index=idf.index)
    common = lab.index.intersection(ref.index)
    ct = pd.crosstab(lab.loc[common], ref.loc[common]).reindex(
        index=range(K), columns=range(K), fill_value=0)
    r, c = linear_sum_assignment(-ct.values)
    renamed = lab.map(dict(zip(ct.index[r], ct.columns[c])))
    return renamed.values, adjusted_rand_score(ref.loc[common], renamed.loc[common])

def draw_heatmap(idf, clusters, cluster_order, palette_order, fname, figsize=(5.6/2.54, 8/2.54),
                 save_csv=True):
    idf = idf.copy()
    genes_idx = [c.split("_")[0] for c in idf.index]
    idf["WT-LOF-Deplete"] *= -1
    kmeans_clustered_heatmap(
        idf, K, clusters, genes_idx, screen_order=screen_order,
        gene_palette=dict(gene_palette), gene_list=gene_list, cluster_order=cluster_order,
        order_col="Q575R-LOF-Enrich",
        axis=replace(AXIS, title="Screens", yticks=[]),
        style=replace(CLUSTEREDHEATMAPSTYLE, default_color=hex_to_rgb("#EEEEEE"),
                      col_width=0.04, heatmap_width=0.8, palette=palette_order,
                      heatmap_kws={"cmap": "bwr_r", "center": 0, "vmin": -10, "vmax": 15,
                                   "cbar": False, "rasterized": True}),
        output=replace(OUTPUT, figsize=figsize, save=True, show=True,
                       path=str(outdir("ClusteredHeatmaps") / fname)))
    if save_csv:
        out = idf[screen_order].copy(); out["WT-LOF-Deplete"] *= -1; out["clusters"] = clusters
        out.to_csv(outdir("ClusteredHeatmaps") / f"{fname}.csv")

editor_cfg = {
    "ABE": (["Q575R_LOF_positiveselection_ABE_zscore", "WT_LOF_dropout_ABE_zscore",
             "Abundance_GFP-EZH2_ABE_zscore", "Abundance_GFP-EED_ABE_zscore"],
            [2, 3, 5, 1, 4, 0]),
    "CBE": (["Q575R_LOF_positiveselection_CBE39_zscore", "WT_LOF_dropout_CBE6b_zscore",
             "Abundance_GFP-EZH2_CBE6b_zscore", "Abundance_GFP-EED_CBE6b_zscore"],
            [5, 1, 0, 4, 2, 3]),
}
cutoff_results = {}
for ed, (cols, corder) in editor_cfg.items():
    pal = {c: hex_to_rgb(tab20[i]) for i, c in enumerate(corder)}
    idf = heatmap_input(cols, hit_thr=config.LOF_STABILITY_CUTOFF)
    raw = kmeans_pca_scatterplot(idf, k=K, random_state=config.KMEANS_RANDOM_STATE)
    clusters, ari = align_to_reference(idf, raw, ed, corder)
    print(f"{ed} cutoff-3: {len(idf)} guides, ARI vs PRC2_Screens_KMeans_Clusters.xlsx = {ari:.4f}")
    cutoff_results[ed] = (idf, clusters, corder, pal)
    draw_heatmap(idf, clusters, corder, pal, f"{ed}_ClusteredHeatmap_K6_Raw_Cutoff3")
print("Clustered heatmaps done")


## 8. Stability-related analysis: forest plots and violin plots

Stability heatmaps/boxplots

The forest-plot / odds-ratio subsection depends on `mutation_level_data.tsv`

The violin plots use the sgRNA-level tables

In [ ]:
# Forest plots + odds-ratio table + histograms + substitution matrix.
try:
    mut = data_loading.load_mutation_level_data()
    or_tsv = outdir("Stability") / "odds_ratio_test_2.0_ALL.tsv"
    structural_enrichment.get_mutation_level_OR_table_tsv(mut, out_tsv=or_tsv)
    structural_enrichment.plot_all_odds_ratio_manuscript_mainfigure(
        or_tsv, out_svg=outdir("Stability") / "Forest_plot_Main.svg")
    structural_enrichment.plot_all_odds_ratio_manuscript_suppl(
        or_tsv, out_dir=outdir("Stability"))
    structural_enrichment.plot_sub_matrix_on_all_merged(
        or_tsv, out_svg=outdir("Stability") / "Substitution_matrix.svg")
    structural_enrichment.plot_distplot_on_structural_regions_v4(
        mut, out_svg=outdir("Stability") / "Histogram_structural_region",
        axis=replace(AXIS, xlabel="GFP-MAX ZScore", xlim=(-3, 12), ylim=(0.0, 1.6),
                     xticks=[-3, 0, 3, 6, 9, 12], yticks=[0, 0.8, 1.6]),
        histtype="bar", bins=70)
    print("Forest / odds-ratio / histogram / substitution-matrix figures done")
except FileNotFoundError as e:
    print("SKIPPING odds-ratio / forest-plot / histogram / substitution-matrix subsection.")
    print("Reason:", str(e).split(". ")[0] + ".")
    print("This is expected until mutation_level_data.tsv is supplied (see errors.md E-02).")

In [ ]:
# Split-violin plots of stability predictors
abe = data_loading.load_sgrna_level("ABE"); abe["editor"] = "ABE"
cbe = data_loading.load_sgrna_level("CBE"); cbe["editor"] = "CBE"
both = pd.concat([abe, cbe])
for name, dd in [("merged", both)]:
    structural_enrichment.get_violin_plot_sgrna_level_v2(
        dd, out_svg=outdir("Stability") / f"Violin_plot_{name}.svg")
print("Violin plots done")

## 9. PWES section

3D-proximity (PWES) clustering against the 6WKR structure.

In [ ]:
# Build the amino-acid-level score table for the LOF+Stability run
q = data_loading.load_q575r_screen(); w = data_loading.load_wt_screen()
s = data_loading.load_stability_screen()
keys = ["sgRNA ID", "sgRNA sequence"]
PWES_TYPES = ["Missense", "Silent", "No Mutation"]

def _lof_subset():
    abe = w[keys + ["Gene Symbol", "AtoG_pos", "AtoG_muttype", "WT_LOF_dropout_ABE_zscore"]].merge(
        q[keys + ["Q575R_LOF_positiveselection_ABE_zscore"]], on=keys, how="left")
    abe = abe[abe["AtoG_pos"] > 0].copy(); abe["pos"] = abe["AtoG_pos"].astype(int)
    abe = abe.rename(columns={"AtoG_muttype": "muttype"}); abe["editor"] = "ABE"
    abe["WT_LOF_dropout_ABE_zscore"] *= -1
    abe["LOF-Score"] = abe[["Q575R_LOF_positiveselection_ABE_zscore",
                            "WT_LOF_dropout_ABE_zscore"]].max(axis=1)
    cbe = w[keys + ["Gene Symbol", "CtoT_pos", "CtoT_muttype", "WT_LOF_dropout_CBE6b_zscore",
                    "WT_LOF_dropout_CBE39_zscore"]].merge(
        q[keys + ["Q575R_LOF_positiveselection_CBE39_zscore"]], on=keys, how="left")
    cbe = cbe[cbe["CtoT_pos"] > 0].copy(); cbe["pos"] = cbe["CtoT_pos"].astype(int)
    cbe = cbe.rename(columns={"CtoT_muttype": "muttype"}); cbe["editor"] = "CBE"
    cbe["WT_LOF_dropout_CBE6b_zscore"] *= -1; cbe["WT_LOF_dropout_CBE39_zscore"] *= -1
    cbe["LOF-Score"] = cbe[["Q575R_LOF_positiveselection_CBE39_zscore",
                            "WT_LOF_dropout_CBE39_zscore", "WT_LOF_dropout_CBE6b_zscore"]].max(axis=1)
    c = ["sgRNA ID", "sgRNA sequence", "Gene Symbol", "pos", "muttype", "editor", "LOF-Score"]
    df = pd.concat([abe[c], cbe[c]])
    return df[df["muttype"].isin(PWES_TYPES)]

def _stability_hit_seqs():
    """Guides with Stability-Enrich > 2.0 (NB1 cell 80) -- excluded from LOF_Stability.
    Unchanged from the previously validated port (reproduces LOF_Stab_Clusters
    exactly, errors.md re-check 2); unlike a per-residue subset it applies no
    position filter."""
    abe = s[keys + ["Gene Symbol", "AtoG_muttype", "Abundance_GFP-EZH2_ABE_zscore",
                    "Abundance_GFP-EED_ABE_zscore"]].rename(columns={"AtoG_muttype": "muttype"})
    cbe = s[keys + ["Gene Symbol", "CtoT_muttype", "Abundance_GFP-EZH2_CBE6b_zscore",
                    "Abundance_GFP-EED_CBE6b_zscore"]].rename(columns={"CtoT_muttype": "muttype"})
    abe["Stability-Enrich"] = abe[["Abundance_GFP-EZH2_ABE_zscore",
                                   "Abundance_GFP-EED_ABE_zscore"]].max(axis=1)
    cbe["Stability-Enrich"] = cbe[["Abundance_GFP-EZH2_CBE6b_zscore",
                                   "Abundance_GFP-EED_CBE6b_zscore"]].max(axis=1)
    both = pd.concat([abe[["sgRNA sequence", "Gene Symbol", "muttype", "Stability-Enrich"]],
                      cbe[["sgRNA sequence", "Gene Symbol", "muttype", "Stability-Enrich"]]])
    both = both[both["muttype"].isin(PWES_TYPES)]
    return both[both["Stability-Enrich"] > 2.0]

def _to_residues(df, score_col):
    amino = df.groupby(["Gene Symbol", "pos"]).agg({
        score_col: "max",
        "sgRNA ID": lambda x: "; ".join(x.astype(str).unique())}).reset_index()
    amino = amino[amino["Gene Symbol"].isin(["EZH2", "EED", "SUZ12"])]
    return pymol_export.remap_to_pdb_numbering(amino, pos_col="pos", shift=True)  # E-11

lof = _lof_subset()
stab = _stability_hit_seqs()
lof_excl = lof.merge(stab[["sgRNA sequence", "Gene Symbol", "Stability-Enrich"]],
                     how="left", on=["sgRNA sequence", "Gene Symbol"])
lof_excl = lof_excl[lof_excl["Stability-Enrich"].isna()]  # exclude stability-driven guides

pwes_runs = {
    "PWES-6WKR-LOF_Stability": (_to_residues(lof_excl, "LOF-Score"), "LOF-Score", "LOF_Stab",
                                {26: 1, 27: 0, 28: 2, 29: 3, 30: 4, 31: 5, 32: 6, 33: 9, 34: 8},
                                (-4, 10)),
}
for prefix, (amino, *_rest) in pwes_runs.items():
    amino.to_csv(outdir("PWES") / f"{prefix}_input_dataframe.csv")
    print(prefix, "input residues:", len(amino))
amino = pwes_runs["PWES-6WKR-LOF_Stability"][0]  # used again by the PyMOL section

In [ ]:
# Run PWES clustering against 6WKR for each score
pwes_results = {}
for prefix, (amino_in, score_col, ref_kind, _, _) in pwes_runs.items():
    res = pwes_clustering(
        df_scores=amino_in, x_col="pos", scores_col=score_col, pdb_file=str(config.PDB_FILE),
        gene_col="Gene Symbol", gene_map={"EZH2": "C", "EED": "L", "SUZ12": "A",
                                          "AEBP2": "P", "JARID2": "B"},
        tanh_a=1.2, gauss_std=16, dend_t=14,
        out_prefix=prefix, out_dir=str(outdir("PWES")) + "/")
    pwes_results[prefix] = res
    df_clus_r = res[3]
    ref = data_loading.load_pwes_clusters(ref_kind).set_index("label")["cl_new"]
    common = df_clus_r.set_index("label")["cl_new"]
    common = common.loc[common.index.intersection(ref.index)]
    print(f"{prefix}: {df_clus_r['cl_new'].nunique()} clusters; ARI vs "
          f"PRC2_Screens_PWES_Clusters.xlsx/{ref_kind} = "
          f"{adjusted_rand_score(ref.loc[common.index], common):.4f}; identical labels: "
          f"{(ref.loc[common.index] == common).all()}")


In [ ]:
# PWES colours + histogram + boxplots
palette = sns.color_palette(palette="tab20")
dark_palette = [x for i, x in enumerate(palette) if i % 2 == 0]
light_palette = [x for i, x in enumerate(palette) if i % 2 == 1]
basic_color = (0.8, 0.8, 0.8)

def pwes_colors(n_clusters, highlight):
    light = [light_palette[highlight[c]] if c in highlight else basic_color
             for c in range(1, n_clusters + 1)]
    dark = [dark_palette[highlight[c]] if c in highlight else basic_color
            for c in range(1, n_clusters + 1)]
    return light, dark

pwes_color_lists = {}
for prefix, (amino_in, score_col, _, highlight, ylim) in pwes_runs.items():
    df_clus_r = pwes_results[prefix][3]
    n_cl = int(df_clus_r["cl_new"].max())
    light_color_list, dark_color_list = pwes_colors(n_cl, highlight)
    pwes_color_lists[prefix] = (light_color_list, dark_color_list)
    pwes_histogram(df_clus=df_clus_r, color_list=light_color_list,
                   axis=replace(AXIS, ylim=(0, 100)), style=replace(PWESHEATMAPSTYLE),
                   output=replace(OUTPUT, figsize=(20/2.54, 4/2.54), save=True,
                                  path=str(outdir("PWES") / prefix)))
    pwes_boxplots(df_clus=df_clus_r, x_col=score_col, color_list=dark_color_list,
                  axis=replace(AXIS, ylim=ylim), style=replace(PWESHEATMAPSTYLE),
                  output=replace(OUTPUT, figsize=(20/2.54, 4/2.54), save=True,
                                 path=str(outdir("PWES") / prefix)))
    for suffix in ("cluster_swarm", "cluster_swarm_boxplot", "cluster_strip"):
        stale = outdir("PWES") / f"{prefix}-{suffix}.svg"
        if os.path.exists(stale):
            os.remove(stale)
print("PWES histogram + boxplots done")


In [ ]:
# PWES heatmap, clustermap, and per-gene scatterplots
from be_scan.figure_plot.figure_pwes_helpers import fillgaps
scat_style = replace(SCATTERSTYLE,
    transparent_kws={"s": 5, "linewidth": 0.1, "edgecolor": "black", "color": "#D3D3D3", "alpha": 0.6},
    opaque_kws={"s": 5, "linewidth": 0.25, "edgecolor": "black", "label": "Cluster"})
for prefix, (amino_in, score_col, _, _, ylim) in pwes_runs.items():
    df_gauss, df_pwes_sorted, df_pwes_unsorted, df_clus_r, aas_dict, link = pwes_results[prefix]
    light_color_list, _ = pwes_color_lists[prefix]
    dh = df_pwes_sorted.copy(); dh[dh <= 0] = 0.0
    dh = fillgaps(dh, gene_bounds={"C": (1, 751), "X": (1, 3), "L": (1, 441),
                                   "Y": (1, 3), "A": (1, 739)})
    pwes_heatmap(dh, mask_on=True, blackout=["X", "Y"],
                 axis=replace(AXIS, xlabel=None, ylabel=None, xticks=[], yticks=[]),
                 style=replace(PWESHEATMAPSTYLE),
                 output=replace(OUTPUT, figsize=(8/2.54, 8/2.54), save=True,
                                path=str(outdir("PWES") / prefix)))
    du = df_pwes_unsorted.copy(); du[du <= 0] = 0.0
    pwes_clustermap(du, link, df_clus_r, color_list=light_color_list,
                    axis=replace(AXIS, xlabel=None, ylabel=None, xticks=[], yticks=[]),
                    style=replace(PWESHEATMAPSTYLE),
                    output=replace(OUTPUT, figsize=(8.5/2.54, 8.5/2.54), save=True,
                                   path=str(outdir("PWES") / prefix)))
    n_cl = int(df_clus_r["cl_new"].max())
    ytk = list(range(ylim[0], ylim[1] + 1, 2))
    for gene in ["EZH2", "EED", "SUZ12"]:
        pwes_scatterplots(
            df_clus_r[df_clus_r["Gene Symbol"] == gene], "pos", score_col, light_color_list,
            max_cluster=n_cl,
            axis=replace(AXIS, xlabel=None, ylabel="sgRNA Z-Score",
                         xlim=(1, xmax_dict[gene]+10), xticks=[1, xmax_dict[gene]],
                         ylim=ylim, yticks=ytk),
            style=scat_style, domain=replace(DOMAIN, dom_setting=domains_all[gene]),
            output=replace(OUTPUT, figsize=(3.5/2.54, 2.5/2.54), save=True,
                           path=str(outdir("PWES") / f"{prefix}-{gene}")))
        highlight = pwes_runs[prefix][3]
        for cluster in range(1, n_cl + 1):
            if cluster in highlight:
                continue
            stale = outdir("PWES") / f"{prefix}-{gene}_subscatter_Cluster{cluster}.svg"
            if os.path.exists(stale):
                os.remove(stale)
print("PWES heatmap / clustermap / scatterplots done")


In [ ]:
# PWES cluster significance (cluster-WAP permutation test) + PyMOL cluster script.
from code import pwes_significance
from be_scan.figure_plot.figure_pwes_plot import generate_pymol_script
for prefix in pwes_runs:
    df_gauss, _, _, df_clus_r, aas_dict, _ = pwes_results[prefix]
    df_pair = pd.read_csv(outdir("PWES") / f"{prefix}_df_pairwise.csv", index_col=0)
    df_wap, df_shuf, df_sum, df_rev = pwes_significance.pwes_permutation_test(
        df_pair, df_gauss, df_clus_r, n_perm=config.PWES_N_PERMUTATIONS)
    df_wap.to_csv(outdir("PWES") / f"{prefix}_df_wap.csv")
    df_shuf.to_csv(outdir("PWES") / f"{prefix}_df_pwes_shuffled_sums.csv")
    df_sum.to_csv(outdir("PWES") / f"{prefix}_df_summary.csv")
    df_rev.to_csv(outdir("PWES") / f"{prefix}_df_summary_revised.csv")
    pwes_significance.plot_wap_histograms(df_wap, outdir("PWES") / f"{prefix}_WAP_histograms.png")
    generate_pymol_script(str(outdir("PWES") / f"{prefix}_aas_dict_FullPDB.pml"),
                          str(config.PDB_FILE), aas_dict, pwes_color_lists[prefix][0])
    print(f"{prefix}: {int(df_rev['Signficiant at 0.05'].sum())}/{len(df_rev)} clusters "
          "significant (Bonferroni 0.05)")


## 10. PyMOL export

Per-residue `.txt` files for b-factor coloring in PyMOL

In [ ]:
pe = pymol_export
pdb_genes = ["EZH2", "EED", "SUZ12", "AEBP2", "JARID2"]
scaffold = pe.build_complete_scaffold(pdb_genes)
types = ["Missense", "Silent", "No Mutation"]

# --- Stability scores ---
def _stab_filtered(zcol, pos_col, mut_col):
    d = data_loading.load_stability_screen()
    d = d[d[mut_col].isin(types)]
    return pe.filter_data(d, zcol, pos_col)

abe_ezh2 = _stab_filtered("Abundance_GFP-EZH2_ABE_zscore", "AtoG_pos", "AtoG_muttype")
cbe_ezh2 = _stab_filtered("Abundance_GFP-EZH2_CBE6b_zscore", "CtoT_pos", "CtoT_muttype")
abe_eed = _stab_filtered("Abundance_GFP-EED_ABE_zscore", "AtoG_pos", "AtoG_muttype")
cbe_eed = _stab_filtered("Abundance_GFP-EED_CBE6b_zscore", "CtoT_pos", "CtoT_muttype")
value_col = "Abundance_GFP-EZH2_ABE_zscore"
for df_in, name in [
        (abe_ezh2.rename(columns={"Abundance_GFP-EZH2_ABE_zscore": "score"}), "ABE-EZH2"),
        (cbe_ezh2.rename(columns={"Abundance_GFP-EZH2_CBE6b_zscore": "score"}), "CBE6-EZH2"),
        (abe_eed.rename(columns={"Abundance_GFP-EED_ABE_zscore": "score"}), "ABE-EED"),
        (cbe_eed.rename(columns={"Abundance_GFP-EED_CBE6b_zscore": "score"}), "CBE6-EED")]:
    collapsed = pe.merge_full_df(df_in, scaffold, "score")
    pe.export_scores_to_txt(collapsed, "score", pdb_genes, outdir("PyMol_txt"),
                            "stability-{gene}-" + name + ".txt")
print("Stability PyMOL txt written")

In [ ]:
# --- LOF scores ---
w = data_loading.load_wt_screen(); q = data_loading.load_q575r_screen()
def _lof_filtered(base, zcol, pos_col, mut_col, neg=False):
    d = base[base[mut_col].isin(types)].copy()
    if neg:
        d[zcol] = -1 * d[zcol]
    d = d.rename(columns={zcol: "score", pos_col: "pos"})
    return pe.filter_data(d, "score", "pos")

parts = {
    "WT-ABE": _lof_filtered(w, "WT_LOF_dropout_ABE_zscore", "AtoG_pos", "AtoG_muttype", neg=True),
    "WT-CBE39": _lof_filtered(w, "WT_LOF_dropout_CBE39_zscore", "CtoT_pos", "CtoT_muttype", neg=True),
    "WT-CBE6": _lof_filtered(w, "WT_LOF_dropout_CBE6b_zscore", "CtoT_pos", "CtoT_muttype", neg=True),
    "Q575R-ABE": _lof_filtered(q, "Q575R_LOF_positiveselection_ABE_zscore", "AtoG_pos", "AtoG_muttype"),
    "Q575R-CBE39": _lof_filtered(q, "Q575R_LOF_positiveselection_CBE39_zscore", "CtoT_pos", "CtoT_muttype"),
}
for name, df_in in parts.items():
    collapsed = pe.merge_full_df(df_in, scaffold, "score")
    pe.export_scores_to_txt(collapsed, "score", pdb_genes, outdir("PyMol_txt"),
                            "lof-{gene}-" + name + ".txt")
print("LOF PyMOL txt written")

In [ ]:
# --- MTF2 scores ---
mtf2 = data_loading.load_mtf2_screen()
mtf2_scaffold = pe.build_complete_scaffold(["MTF2"])
for zcol, name in [("WT_DMSO_D0_Wk3_zscore", "WT-DMSO-D0"),
                   ("Q575R_DMSO_D0_Wk3_zscore", "Q575R-DMSO-D0")]:
    d = mtf2[mtf2["AtoG_muttype"].isin(types)].copy()
    d[zcol] = -1 * d[zcol]
    d = pe.filter_data(d, zcol, "aa_pos")
    collapsed = pe.merge_full_df(d, mtf2_scaffold, zcol)
    pe.export_scores_to_txt(collapsed, zcol, ["MTF2"], outdir("PyMol_txt"),
                            "MTF2-" + name + ".txt", apply_pdb_remap=False)
print("MTF2 PyMOL txt written")

In [ ]:
# --- LOF-Stability adjusted scores ---
lof76 = _lof_subset()
stab76 = _stability_hit_seqs()
lof76 = lof76.merge(stab76[["sgRNA sequence", "Gene Symbol", "Stability-Enrich"]],
                    how="left", on=["sgRNA sequence", "Gene Symbol"])
lof76 = lof76[lof76["Stability-Enrich"].isna()]
amino76 = lof76.groupby(["Gene Symbol", "pos"]).agg({"LOF-Score": "max"}).reset_index()
amino76["LOF-Score"] = amino76["LOF-Score"].clip(lower=0)
full = pe.build_complete_scaffold(pdb_genes).merge(amino76, on=["Gene Symbol", "pos"], how="left")
full["LOF-Score"] = full["LOF-Score"].fillna(0)
pe.export_scores_to_txt(full, "LOF-Score", pdb_genes, outdir("PyMol_txt"),
                        "LOF-Stability-{gene}.txt")
print("LOF-Stability adjusted PyMOL txt written")

import glob
print("Total PyMOL .txt files:", len(glob.glob(str(OUTPUT_DIR / 'PyMol_txt' / '*.txt'))))